<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Step 8</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Step 8: Engines</h1>
<p style="margin:.55em 0">Steps 6 and 7 built one engine: speech-to-speech over our own websocket. Two more share its contract (audio in, audio
out, skills through <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">run_tool</code>, the same notes for the page), picked by <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">[engine] kind</code> in config:</p>
<div style="display:flex;flex-direction:column;align-items:flex-start;gap:4px;margin:.8em 0">
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>realtime</b>: browser → our server → websocket → OpenAI. Simple; but audio rides one TCP stream.</div>
  <div style="border:2px solid #6366f1;background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>webrtc</b>: the browser's audio goes straight to OpenAI over WebRTC (UDP: a lost packet is a
  20 ms glitch, not a frozen reply). Our server creates the call with its key and attaches a side connection that runs
  the tools and guards.</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>cascade</b>: speech-to-text → a text model with our tools → text-to-speech, each a provider module
  (OpenAI today; Sarvam or ElevenLabs are one file each). Slower, but every step is swappable and text can be fixed in
  code before the model reads it.</div>
</div>

</div>

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">WebRTC</h2>
<p style="margin:.55em 0">The page sends its SDP offer over our websocket; <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">create_call</code> posts it to <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">/v1/realtime/calls</code> with the
session (prompt, tools, voice, turn detection); the answer goes back to the page; <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">attach</code> opens the side
connection by call id. Barge-in happens inside OpenAI, which knows what was played. The call is hung up when the page leaves.</p>

</div>

In [1]:
%%writefile build/app/webrtc.py
"""The WebRTC engine: the browser's audio goes straight to OpenAI over WebRTC; our server keeps the control.

    browser ──WebRTC audio (UDP, loss-tolerant)──► OpenAI Realtime
       │ offer / answer, page events                 ▲
       ▼                                             │ side connection: tools, guards, notes
    our server ──────────────────────────────────────┘

Why: over a websocket, audio rides one TCP stream; one lost packet stalls everything behind it while TCP backs off
(measured: 3 in 10 long replies froze, two came back after exactly 61.5 s). WebRTC carries audio as small UDP
packets: a lost one is a 20 ms glitch, not a frozen reply. The side connection carries only small events.

Flow: the page sends its SDP offer over our /ws; we create the call with OUR key (POST /v1/realtime/calls with
the session config: prompt, tools, voice, turn detection), hand back the answer, then attach to the same call
(wss .../realtime?call_id=...) to run tools and report events. The key never reaches the browser. Barge-in and
truncation happen inside OpenAI (it knows what was played). The filler clip is skipped here: the model's own
"let me check" covers tool calls.
"""
import asyncio
import json
import logging
import time
import urllib.request
import uuid

import websockets

from app import realtime

log = logging.getLogger(__name__)
MAX_SDP = 20_000  # bytes; a real offer is ~3-6 KB


def rest_base(s) -> str:  # wss://api.openai.com/v1/realtime -> https://api.openai.com/v1
    return s.openai.url.replace("wss://", "https://", 1).replace("ws://", "http://", 1).rsplit("/realtime", 1)[0]


def call_config(s, instructions: str, tools: list, vad: dict | None) -> dict:
    """The session, as sent with the call (audio format is WebRTC's own)."""
    return {
        "type": "realtime", "model": s.openai.model, "instructions": instructions,
        **({"tools": tools, "tool_choice": "auto"} if tools else {}),
        "max_output_tokens": s.openai.max_output_tokens,
        "audio": {"input": {"turn_detection": vad,
                            "transcription": {"model": s.openai.transcribe_model} if s.openai.transcribe_model else None},
                  "output": {"voice": s.openai.voice}},
    }


def _post_call(s, key, offer_sdp: str, session: dict) -> tuple[str, str]:
    boundary = uuid.uuid4().hex
    part = lambda name, ctype, body: (f"--{boundary}\r\nContent-Disposition: form-data; name=\"{name}\"\r\n"
                                      f"Content-Type: {ctype}\r\n\r\n{body}\r\n")
    body = (part("sdp", "application/sdp", offer_sdp) + part("session", "application/json", json.dumps(session))
            + f"--{boundary}--\r\n").encode()
    req = urllib.request.Request(f"{rest_base(s)}/realtime/calls", data=body, method="POST", headers={
        "Authorization": f"Bearer {key}", "Content-Type": f"multipart/form-data; boundary={boundary}"})
    with urllib.request.urlopen(req, timeout=20) as r:
        return r.read().decode(), r.headers["Location"].rstrip("/").rsplit("/", 1)[-1]


async def create_call(s, key, offer_sdp: str, session: dict) -> tuple[str, str]:
    """POST /v1/realtime/calls -> (answer SDP, call id)."""
    return await asyncio.to_thread(_post_call, s, key, offer_sdp, session)


def _hangup(s, key, call_id):
    req = urllib.request.Request(f"{rest_base(s)}/realtime/calls/{call_id}/hangup", data=b"", method="POST",
                                 headers={"Authorization": f"Bearer {key}"})
    urllib.request.urlopen(req, timeout=10).close()


async def attach(s, key, call_id):
    """The side connection to a running call."""
    return await websockets.connect(f"{s.openai.url}?call_id={call_id}", max_size=None,
                                    additional_headers={"Authorization": f"Bearer {key}"})


async def converse(s, key, offers: asyncio.Queue, send_answer, chunks, seconds: float, instructions: str,
                   tools: list, run_tool, vad: dict | None = None, on_note=None, on_ready=None,
                   stop: asyncio.Event | None = None, max_resumes: int = 2, create=None, connect=None,
                   hangup=None) -> list[tuple]:
    """One WebRTC conversation. offers: the page's SDP offer arrives here; send_answer(sdp) returns OpenAI's.
    chunks: the page's message stream, drained only to notice it leaving. create/connect: injectable for tests."""
    create, connect = create or create_call, connect or attach
    hangup = hangup or (lambda call_id: asyncio.to_thread(_hangup, s, key, call_id))
    t0, log_ = time.perf_counter(), []

    def note(event, detail=""):
        log_.append((round(time.perf_counter() - t0, 2), event, detail))
        log.debug("%s %s %s", *log_[-1])
        if on_note:
            on_note(*log_[-1])

    async def drain():  # ends (or raises) when the page leaves
        async for _ in chunks:
            pass

    leave = asyncio.create_task(drain())
    call_id = None
    try:
        got = asyncio.create_task(asyncio.wait_for(offers.get(), 30))
        await asyncio.wait([got, leave], return_when=asyncio.FIRST_COMPLETED)
        if not got.done():
            got.cancel()
            leave.result()  # the page left before offering: BrowserGone
            return log_
        offer = got.result()
        answer, call_id = await create(s, key, offer, call_config(s, instructions, tools, vad))
        await send_answer(answer)
        note("call", call_id)

        speaking, resumes = False, 0

        async def events():
            nonlocal speaking, resumes
            ws = await connect(s, key, call_id)
            if on_ready:
                on_ready()
            while True:
                try:
                    async for raw in ws:
                        e = json.loads(raw)
                        kind = e["type"]
                        if kind == "error":
                            raise RuntimeError(e["error"])
                        if kind == "input_audio_buffer.speech_started":
                            note(kind, "")
                            if speaking:
                                note("barge_in", "OpenAI stops and truncates the reply")
                        elif kind == "input_audio_buffer.speech_stopped":
                            note(kind)
                        elif kind == "conversation.item.input_audio_transcription.completed":
                            note("user_said", e["transcript"].strip())
                        elif kind == "conversation.item.input_audio_transcription.failed":
                            note("user_unheard", e.get("error", {}).get("message", ""))
                        elif kind == "output_audio_buffer.started":
                            speaking = True
                            note("first_audio_delta")
                        elif kind in ("output_audio_buffer.stopped", "output_audio_buffer.cleared"):
                            speaking = False
                            note("speaking_done")
                        elif kind == "response.output_audio_transcript.done":
                            note("model_said", e["transcript"])
                        elif kind == "response.done":
                            note(kind, e["response"]["status"])
                            for name, args, out in await realtime.answer_tool_calls(ws, e["response"], run_tool):
                                note("tool_call", f"{name}({' '.join(args.split())}) -> {json.dumps(out)}")
                    return
                except realtime.DROPPED as exc:  # the side link died; the call (audio) carries on: re-attach
                    if resumes == max_resumes:
                        raise
                    resumes += 1
                    note("RESUME", f"control link dropped ({type(exc).__name__}): re-attaching ({resumes}/{max_resumes})")
                    ws.transport.abort()
                    ws = await connect(s, key, call_id)

        runner = asyncio.create_task(events())
        waits = [runner, leave] + ([asyncio.create_task(stop.wait())] if stop else [])
        done, pending = await asyncio.wait(waits, timeout=seconds, return_when=asyncio.FIRST_COMPLETED)
        for t in pending:
            t.cancel()
        await asyncio.gather(*pending, return_exceptions=True)
        for t in done:
            t.result()
        return log_
    finally:
        leave.cancel()
        if call_id:
            try:  # end the call at OpenAI too: stops billing at once instead of at its own timeout
                await hangup(call_id)
            except Exception as e:
                log.warning("hangup %s failed: %r", call_id, e)


Writing build/app/webrtc.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Providers</h2>
<p style="margin:.55em 0">A provider is a module with any of three classes: <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">STT</code>, <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">LLM</code>, <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">TTS</code>. <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">load</code> finds them by
the names in config and fails at startup if one is missing.</p>

</div>

In [2]:
%%writefile build/app/providers/__init__.py
"""Providers for the cascade engine (app/cascade.py): speech-to-text, a text model, text-to-speech.

A provider is a module here (app/providers/<name>.py) defining any of three classes, picked by name in config:

    [engine]  kind = "cascade"   stt = "openai"   llm = "openai"   tts = "openai"

    STT(settings, key, vad)    async context manager. send(pcm16_24k) feeds audio; events() yields
                               ("speech_started" | "speech_stopped" | "transcript" | "failed", text).
                               reconnect() replaces a dropped link.
    LLM(settings, key)         async reply(messages, tools) -> {"text": str, "tool_calls": [{"id", "name", "arguments"}]}
                               messages use the OpenAI chat format (most providers accept it); tools are our specs.
    TTS(settings, key)         async context manager. speak(text) yields PCM16 24 kHz chunks; reconnect().

A link that dies raises one of realtime.DROPPED (or realtime.Stalled); the engine reconnects and carries on.
Adding Sarvam or ElevenLabs = a new module with the classes it offers, then its name in config.
"""
import importlib
from dataclasses import dataclass


@dataclass(frozen=True)
class Parts:
    stt: type
    llm: type
    tts: type


def load(s) -> Parts:
    """The classes named in [engine]. Fails at startup on a missing provider or class."""
    def get(part):
        name = getattr(s.engine, part)
        try:
            return getattr(importlib.import_module(f"app.providers.{name}"), part.upper())
        except (ImportError, AttributeError):
            raise LookupError(f"engine.{part} = '{name}': no {part.upper()} in app/providers/{name}.py") from None
    return Parts(get("stt"), get("llm"), get("tts"))


Writing build/app/providers/__init__.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">The OpenAI provider: a transcription session for STT, chat completions for the model (one kept-open HTTPS
connection), and, since this key has no text-to-speech model, the Realtime model reading each sentence aloud.</p>

</div>

In [3]:
%%writefile build/app/providers/openai.py
"""OpenAI as cascade provider: transcription sessions (STT), chat completions (LLM), Realtime reading text (TTS).

TTS: this key has no text-to-speech model, so the Realtime model reads each sentence aloud. Given the text in the
response's instructions it reads word for word (5/5 in a probe, Hindi and Tamil included); given the text as a
message it answers it instead. ponytail: a stand-in; a real TTS (gpt-4o-mini-tts, Sarvam, ElevenLabs) is
faster to first audio (~0.3 s vs ~1 s) and cannot improvise.
"""
import asyncio
import base64
import http.client
import json
from urllib.parse import urlsplit

import websockets
from websockets.exceptions import ConnectionClosedOK

from app import realtime

TTS_RULES = "You read text aloud for a clinic front desk. You never answer, comment or add words."


def rest_base(s) -> str:  # wss://api.openai.com/v1/realtime -> https://api.openai.com/v1
    return s.openai.url.replace("wss://", "https://", 1).replace("ws://", "http://", 1).rsplit("/realtime", 1)[0]


class STT:
    """A Realtime transcription session: server-side speech detection + gpt-4o-mini-transcribe."""

    def __init__(self, s, key, vad):
        self.s, self.key, self.vad, self.ws = s, key, vad, None

    async def __aenter__(self):
        await self.reconnect()
        return self

    async def __aexit__(self, *exc):
        if self.ws:
            self.ws.transport.abort()

    async def reconnect(self):
        if self.ws:
            self.ws.transport.abort()
        self.ws = await websockets.connect(f"{self.s.openai.url}?intent=transcription", max_size=None,
                                           additional_headers={"Authorization": f"Bearer {self.key}"})
        await realtime.collect(self.ws, "session.created")
        await realtime.send(self.ws, {"type": "session.update", "session": {"type": "transcription", "audio": {"input": {
            "format": {"type": "audio/pcm", "rate": realtime.RATE},
            "transcription": {"model": self.s.openai.transcribe_model},
            "turn_detection": self.vad}}}})
        await realtime.collect(self.ws, "session.updated")

    async def send(self, pcm: bytes):
        await realtime.send(self.ws, {"type": "input_audio_buffer.append", "audio": realtime.b64(pcm)})

    async def events(self):
        async for raw in self.ws:
            e = json.loads(raw)
            kind = e["type"]
            if kind == "error":
                raise RuntimeError(e["error"])
            if kind == "input_audio_buffer.speech_started":
                yield "speech_started", ""
            elif kind == "input_audio_buffer.speech_stopped":
                yield "speech_stopped", ""
            elif kind == "conversation.item.input_audio_transcription.completed":
                yield "transcript", e["transcript"].strip()
            elif kind == "conversation.item.input_audio_transcription.failed":
                yield "failed", e.get("error", {}).get("message", "")
        raise ConnectionClosedOK(None, None)  # the link ended: let the engine reconnect


class LLM:
    """Chat completions with tools (gpt-4o-mini by default: [engine] llm_model)."""

    def __init__(self, s, key):
        self.s, self.key, self.conn = s, key, None

    def _post(self, body: dict) -> dict:
        """One kept-open HTTPS connection per conversation (calls are sequential): saves a TLS handshake per call."""
        url = urlsplit(rest_base(self.s))
        for attempt in range(2):  # a kept-open connection the server closed meanwhile: open a new one once
            if self.conn is None:
                cls = http.client.HTTPSConnection if url.scheme == "https" else http.client.HTTPConnection
                self.conn = cls(url.netloc, timeout=self.s.engine.llm_timeout_s)
            try:
                self.conn.request("POST", f"{url.path}/chat/completions", body=json.dumps(body),
                                  headers={"Authorization": f"Bearer {self.key}", "Content-Type": "application/json"})
                r = self.conn.getresponse()
                data = r.read()
            except (http.client.HTTPException, OSError):
                self.conn.close()
                self.conn = None
                if attempt:
                    raise
                continue
            if r.status != 200:
                raise RuntimeError(f"chat completions HTTP {r.status}: {data[:300]!r}")
            return json.loads(data)

    async def reply(self, messages: list, tools: list) -> dict:
        body = {"model": self.s.engine.llm_model, "messages": messages, "max_completion_tokens": 400}
        if tools:  # our specs are Realtime-style; chat completions nests them under "function"
            body["tools"] = [{"type": "function", "function": {k: t[k] for k in ("name", "description", "parameters")}}
                             for t in tools]
        msg = (await asyncio.to_thread(self._post, body))["choices"][0]["message"]
        return {"text": msg.get("content") or "",
                "tool_calls": [{"id": c["id"], "name": c["function"]["name"], "arguments": c["function"]["arguments"]}
                               for c in msg.get("tool_calls") or []]}


class TTS:
    """The Realtime model reading text aloud, one out-of-band response per sentence."""

    def __init__(self, s, key):
        self.s, self.key, self.ws = s, key, None

    async def __aenter__(self):
        await self.reconnect()
        return self

    async def __aexit__(self, *exc):
        if self.ws:
            self.ws.transport.abort()

    async def reconnect(self):
        if self.ws:
            self.ws.transport.abort()
        self.ws = await realtime.start_session(self.s, self.key, TTS_RULES, [], None)

    async def speak(self, text: str):
        await realtime.send(self.ws, {"type": "response.create", "response": {
            "conversation": "none", "output_modalities": ["audio"], "input": [],
            "instructions": f"Say exactly this line, word for word, and nothing else: {text}"}})
        rid, done = None, False
        try:
            while True:
                try:
                    e = await asyncio.wait_for(realtime.recv(self.ws), self.s.resilience.stall_s)
                except asyncio.TimeoutError:
                    raise realtime.Stalled(f"no audio for {self.s.resilience.stall_s}s") from None
                if e["type"] == "response.created":
                    rid = e["response"]["id"]
                elif e.get("response_id", rid) != rid:  # a cancelled earlier sentence still draining
                    continue
                elif e["type"] == "response.output_audio.delta":
                    yield base64.b64decode(e["delta"])
                elif e["type"] == "response.done" and e["response"]["id"] == rid:
                    done = True
                    return
        finally:
            if not done and rid:  # the caller stopped listening (barge-in): stop generating
                try:
                    await realtime.send(self.ws, {"type": "response.cancel", "response_id": rid})
                except realtime.DROPPED:
                    pass


Writing build/app/providers/openai.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">The cascade</h2>
<p style="margin:.55em 0">One worker answers turns in order. Tool calls always run to the end; a reply is dropped if the caller spoke again
before it was said; talking over a reply stops it and the history keeps only the part heard. <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">spoken_digits</code> turns
numbers said digit by digit, in Hindi, Tamil or English, into digits before the model reads them: the text model got them
wrong on its own.</p>

</div>

In [4]:
%%writefile build/app/cascade.py
"""The cascade engine: speech-to-text -> a text model with our tools -> text-to-speech, each a swappable provider.

Same contract as realtime.converse_resilient: caller audio in (`chunks`), audio out (`speaker`), skills through
`run_tool`, and the same notes for the page (speech_started, user_said, model_said, first_audio_delta, filler,
tool_call, barge_in, response.done, STALL, RESUME). The conversation history is ours, so a dropped provider link
is just reopened: nothing to replay.

Turns: each final transcript is a user message. One worker answers them in order; if the caller speaks again
before a reply is spoken, the reply is dropped and the model answers everything at once. Tool calls always run
to the end (a booking must not be half-done). The caller talking over a reply stops its audio at once, and the
history keeps only the part they heard.
"""
import asyncio
import contextlib
import json
import logging
import re
import time

from app.realtime import BYTES_PER_S, DROPPED, FILLER_TEXT, Stalled

log = logging.getLogger(__name__)
SENTENCE_END = re.compile(r"(?<=[.!?।])\s+")  # । = Devanagari full stop

# Spoken digits, 0-9, as callers say them (formal and everyday forms). ponytail: single digits only ("बारह" = 12 stays)
DIGIT_WORDS = [
    "zero oh शून्य शुन्य जीरो ज़ीरो பூஜ்ஜியம் பூஜ்யம் சைபர் ஜீரோ",
    "one एक ஒன்று ஒன்னு",
    "two दो இரண்டு ரெண்டு",
    "three तीन மூன்று மூணு",
    "four चार நான்கு நாலு",
    "five पाँच पांच ஐந்து அஞ்சு",
    "six छह छः छै छे ஆறு",
    "seven सात ஏழு",
    "eight आठ எட்டு",
    "nine नौ ஒன்பது ஒம்பது",
]
DIGIT = {w: str(d) for d, words in enumerate(DIGIT_WORDS) for w in words.split()}
REPEAT = {"double": 2, "डबल": 2, "டபுள்": 2, "triple": 3, "ट्रिपल": 3, "ட்ரிபிள்": 3}
_d = "|".join(sorted(DIGIT, key=len, reverse=True)) + r"|\d+"
_unit = rf"(?:(?:{'|'.join(REPEAT)})\s*)?(?:{_d})"
_edge = r"\wऀ-ॣ०-෿"  # letters + Indic letters and vowel signs (not the । ॥ full stops)
DIGIT_RUN = re.compile(rf"(?<![{_edge}]){_unit}(?:[\s,–-]+{_unit})*(?![{_edge}])", re.IGNORECASE)
DIGIT_UNIT = re.compile(rf"(?:({'|'.join(REPEAT)})\s*)?({_d})", re.IGNORECASE)


def spoken_digits(text: str) -> str:
    """Numbers said digit by digit, in any language, become digits before the model reads them:
    "नौ आठ चार पाँच" -> "9845", "nine double eight" -> "988". A text model turning number words into digits
    itself gets them wrong (measured: Hindi words -> one digit wrong). Runs shorter than 3 digits stay words."""
    def join(m):
        units = DIGIT_UNIT.findall(m[0])
        if not any(w.lower() in DIGIT for _, w in units):  # already digits ("10, 11"): leave as said
            return m[0]
        digits = "".join(DIGIT.get(w.lower(), w) * REPEAT.get(r.lower(), 1) for r, w in units)
        return digits if len(digits) >= 3 else m[0]
    return DIGIT_RUN.sub(join, text)


def sentences(text: str) -> list[str]:
    return [p for p in SENTENCE_END.split(text.strip()) if p]


async def converse(parts, s, key, chunks, speaker, seconds: float, instructions: str, tools: list, run_tool,
                   vad: dict | None = None, filler_pcm: bytes | None = None, on_note=None, on_ready=None,
                   stop: asyncio.Event | None = None, max_resumes: int = 2) -> list[tuple]:
    """Run one conversation for up to `seconds`. `parts`: providers.Parts (classes; instantiated here)."""
    t0, log_ = time.perf_counter(), []
    messages = [{"role": "system", "content": instructions}]
    waiting: list[str] = []         # transcripts not answered yet
    wake = asyncio.Event()
    turn = 0                        # bumped by every transcript: a reply for an older turn is not spoken
    cut = 0                         # bumped by barge-in: the reply being spoken stops
    speaking = False
    resumes = 0

    def note(event, detail=""):
        log_.append((round(time.perf_counter() - t0, 2), event, detail))
        log.debug("%s %s %s", *log_[-1])
        if on_note:
            on_note(*log_[-1])

    def resume(what, e):
        nonlocal resumes
        if resumes == max_resumes:
            raise e
        resumes += 1
        note("RESUME", f"{what} link dropped ({type(e).__name__}): reconnecting ({resumes}/{max_resumes})")

    llm = parts.llm(s, key)
    async with parts.stt(s, key, vad) as stt, parts.tts(s, key) as tts:

        async def send_loop():  # caller -> speech-to-text
            async for c in chunks:
                with contextlib.suppress(*DROPPED):  # a dropped link loses this 100 ms; listen() reconnects it
                    await stt.send(c)

        async def listen():  # speech-to-text events -> turns
            nonlocal turn, cut
            while True:
                try:
                    async for kind, text in stt.events():
                        if kind == "speech_started":
                            note("input_audio_buffer.speech_started", f"{speaker.buffered_seconds():.2f}s of audio still queued")
                            if speaking or speaker.buffered_seconds() > 0:
                                cut += 1
                                dropped = speaker.clear()
                                note("barge_in", f"dropped {dropped / BYTES_PER_S:.2f}s")
                        elif kind == "speech_stopped":
                            note("input_audio_buffer.speech_stopped")
                        elif kind == "transcript":
                            note("user_said", text)
                            if text:
                                waiting.append(text)
                                turn += 1
                                wake.set()
                        elif kind == "failed":
                            note("user_unheard", text)
                    return
                except DROPPED as e:
                    resume("speech-to-text", e)
                    await stt.reconnect()

        async def think(my_turn) -> str | None:
            """The model's answer (running any tools it asks for), or None if the caller spoke again meanwhile."""
            filled = False
            for _ in range(5):  # tool rounds per answer: a model looping on tools must not hang the call
                for attempt in range(2):  # one retry: a single failed HTTP call should not end the call
                    try:
                        out = await llm.reply(messages, tools)
                        break
                    except Exception as e:
                        if attempt:
                            raise
                        note("STALL", f"text model: {type(e).__name__} {e}")
                if not out["tool_calls"]:
                    return out["text"].strip() if my_turn == turn else None
                messages.append({"role": "assistant", "content": out["text"] or None, "tool_calls": [
                    {"id": c["id"], "type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}}
                    for c in out["tool_calls"]]})
                if filler_pcm and not filled:
                    filled = True
                    speaker.write(filler_pcm)
                    note("filler", f'"{FILLER_TEXT}" while {out["tool_calls"][0]["name"]} runs')
                for c in out["tool_calls"]:  # always run to the end, even if the caller talks meanwhile
                    result = await run_tool(c["name"], c["arguments"])
                    messages.append({"role": "tool", "tool_call_id": c["id"], "content": json.dumps(result)})
                    note("tool_call", f"{c['name']}({' '.join(c['arguments'].split())}) -> {json.dumps(result)}")
            return "Sorry, I could not finish that. Could you say it again?"

        async def say(text, my_cut) -> str:
            """Speak `text` sentence by sentence; returns the part the caller heard."""
            nonlocal speaking
            start, spoken, heard_text = speaker.position, 0, []
            speaking = True
            try:
                for sentence in sentences(text):
                    for attempt in range(2):  # a stalled link loses one sentence at most: say it again
                        try:
                            async with contextlib.aclosing(tts.speak(sentence)) as audio:  # closing = cancel it
                                async for pcm in audio:
                                    if my_cut != cut:
                                        break
                                    if not spoken:  # the page shows the reply as its sound starts
                                        note("first_audio_delta")
                                        note("model_said", text)
                                    speaker.write(pcm)
                                    spoken += len(pcm)
                            break
                        except (Stalled, *DROPPED) as e:
                            note("STALL", f"text-to-speech: {type(e).__name__} {e}")
                            resume("text-to-speech", e)
                            await tts.reconnect()
                    if my_cut != cut:
                        heard = min(max(speaker.played - start, 0), spoken)
                        part = sentence[:len(sentence) * heard // max(spoken, 1)]  # ponytail: by audio share
                        return " ".join(heard_text + [part]).strip() + "…"
                    heard_text.append(sentence)
                return text
            finally:
                speaking = False

        async def worker():  # answers turns in order, one at a time
            while True:
                await wake.wait()
                wake.clear()
                while waiting:
                    messages.append({"role": "user", "content": spoken_digits(" ".join(waiting))})
                    waiting.clear()
                    answer = await think(turn)
                    if answer:
                        heard = await say(answer, cut)
                        messages.append({"role": "assistant", "content": heard})
                    note("response.done", "superseded" if answer is None else "completed")

        if on_ready:
            on_ready()
        tasks = [asyncio.create_task(f()) for f in (send_loop, listen, worker)]
        if stop:
            tasks.append(asyncio.create_task(stop.wait()))
        done, pending = await asyncio.wait(tasks, timeout=seconds, return_when=asyncio.FIRST_COMPLETED)
        for t in pending:
            t.cancel()
        await asyncio.gather(*pending, return_exceptions=True)
        for t in done:
            t.result()  # re-raise a provider error instead of hiding it
    return log_


Writing build/app/cascade.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Try it</h2>

</div>

In [5]:
import sys
sys.path.insert(0, 'build')   # `import app` now means the app you are building

In [6]:
from app.cascade import spoken_digits
[spoken_digits(t) for t in ('मेरा नंबर है नौ आठ चार पाँच शून्य एक दो तीन छः सात।',
                            'நம்பர் ஒன்பது எட்டு நான்கு ஐந்து பூஜ்ஜியம் ஒன்று இரண்டு மூன்று ஆறு ஏழு.',
                            'nine eight four five zero one two three double seven', 'मुझे एक अपॉइंटमेंट चाहिए')]

['मेरा नंबर है 9845012367।',
 'நம்பர் 9845012367.',
 '9845012377',
 'मुझे एक अपॉइंटमेंट चाहिए']

In [7]:
from app import settings, webrtc
s = settings.load(env={'APP__STORE__PATH': 'data/step8.db'})
cfg = webrtc.call_config(s, 'You are a front desk.', [], {'type': 'server_vad'})
cfg['model'], cfg['audio']

('gpt-realtime-mini',
 {'input': {'turn_detection': {'type': 'server_vad'},
   'transcription': {'model': 'gpt-4o-mini-transcribe'}},
  'output': {'voice': 'marin'}})

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Both engines are tested end to end with fakes by the self-check in step 10. <b>Next:</b> <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">09_server.ipynb</code>.</p>

</div>